**Logistic Regression + T2: PCA (95% variance)**

This notebook trains **LR with transformation T2** for 6 values of `C` (6 of our 72 models). It uses the shared splits (`data/processed/`), preprocessing (`src/preprocessing.py`) and evaluation (`src/evaluation.py`), so the results can be merged with the other notebooks. We only use train and validation here; the test set is used once at the end of the project.

**Logistic regression** predicts the probability of an attack as $P(\text{attack}) = \sigma(w^\top x + b)$, where $\sigma(z) = 1 / (1 + e^{-z})$ squashes any number into (0, 1). It flags a flow as an attack when that probability is above 0.5, so the decision boundary $w^\top x + b = 0$ is a straight line (hyperplane) in the feature space it sees.

**Regularization:** sklearn minimizes

$$\frac{1}{2}\lVert w \rVert^2 + C \sum_{i} \text{log-loss}_i$$

`C` sets how much the model cares about fitting the training data compared to keeping the weights small. **Small `C` ---> strong regularization** (small weights, simpler model, risk of underfitting). **Large `C` ---> weak regularization** (fits training data closely, risk of overfitting). `C` plays the role of $1/\lambda$.

**1. Setup**

In [ ]:
import sys
sys.path.append("..") # so we can import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from src.preprocessing import RANDOM_STATE, build_preprocessor, load_split
from src.evaluation import evaluate, plot_sweep, attack_recall, RESULTS

FIGS = RESULTS.parent / "figures"
RESULTS.mkdir(exist_ok=True)

X_train, y_train, cat_train = load_split("train")
X_val, y_val, cat_val = load_split("val")
print("train:", X_train.shape, " val:", X_val.shape)

**2. The transformation: PCA (95% variance)**

The EDA found 16 pairs of features with correlation above 0.9, so many of our 73 features repeat the same information. PCA rotates the feature space so that the new axes (principal components) are uncorrelated and sorted by how much variance they carry, then we keep only as many as needed for 95% of the variance.

- The components are the eigenvectors of the covariance matrix $\Sigma = \frac{1}{N} X^\top X$, and each eigenvalue $\lambda_i$ is the variance along that component
- We keep the smallest $k$ with $\frac{\lambda_1 + \dots + \lambda_k}{\lambda_1 + \dots + \lambda_n} \ge 0.95$

**Why do we do this?**: Fewer, uncorrelated features ---> less redundancy for the linear model. The risk is that PCA never looks at the label, so a low-variance direction that separates attacks could be dropped.

In [ ]:
prep = build_preprocessor().fit(X_train)
Xt_train = prep.transform(X_train)
pca = PCA(random_state=RANDOM_STATE).fit(Xt_train)
cum_var = np.cumsum(pca.explained_variance_ratio_)
k95 = np.argmax(cum_var >= 0.95) + 1
print(f"components needed for 95% variance: {k95} out of {Xt_train.shape[1]}")

plt.plot(range(1, len(cum_var) + 1), cum_var, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.axvline(k95, color="gray", linestyle=":")
plt.xlabel("number of components")
plt.ylabel("cumulative explained variance")
plt.title("PCA explained variance (training split)")
plt.savefig(FIGS / "lr_t2_pca_variance.png", dpi=150, bbox_inches="tight")
plt.show()

What do the first components actually measure? We look at the original features with the largest weights (loadings) in each of the first 3 components.

In [ ]:
names = prep.get_feature_names_out()
loadings = pd.DataFrame(pca.components_[:3].T, index=names, columns=["PC1", "PC2", "PC3"])
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(5).index
    print(pc, "--->", ", ".join(f"{n} ({loadings.loc[n, pc]:+.2f})" for n in top))

PC1 vs PC2 for a sample of training flows, colored by label. PCA never saw the labels, so any separation here comes from the structure of the data alone.

In [ ]:
Z = pca.transform(Xt_train)[:, :2]
sample = np.random.default_rng(RANDOM_STATE).choice(len(Z), 5000, replace=False)
for label, name, color in [(0, "Normal", "tab:blue"), (1, "Attack", "tab:red")]:
    m = y_train.values[sample] == label
    plt.scatter(Z[sample][m, 0], Z[sample][m, 1], s=4, alpha=0.4, color=color, label=name)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.title("Training flows on the first two principal components")
plt.savefig(FIGS / "lr_t2_pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

> **[Your input: transformation]** What did the checks above show? Send me your notes and I'll write them up.

**3. The pipeline**

The full pipeline: shared preprocessing (73 features) ---> PCA keeping 95% of the variance ---> logistic regression.

In [ ]:
def build(C):
    return make_pipeline(build_preprocessor(),
                         PCA(n_components=0.95, random_state=RANDOM_STATE),
                         LogisticRegression(C=C, max_iter=2000))

**4. Regularization sweep**

We train one model for each of the 6 values of `C` (smaller `C` = stronger regularization). Every model is fit on the 24,000 training rows and scored on both train and validation. `f1_gap` = train F1 - validation F1: a big gap means overfitting, while low F1 on both means underfitting.

In [ ]:
GRID = [1e-3, 1e-2, 1e-1, 1, 10, 100]
rows, fitted = [], {} # fitted keeps each trained pipeline, so we don't have to retrain the best one

for value in GRID:
    pipe = build(value)
    rows.append(evaluate(pipe, "LR", "T2", "C", value, X_train, y_train, X_val, y_val))
    fitted[value] = pipe
    print(f"C={value:g}: val F1 = {rows[-1]['val_f1']:.4f}, fit time = {rows[-1]['fit_time_s']}s")

results = pd.DataFrame(rows)
results["f1_gap"] = results["train_f1"] - results["val_f1"]
results.drop(columns=["model", "transformation", "param"]).round(4)

**5. Validation curve**

Left: train F1 (dashed) vs validation F1 (solid). Right: validation precision, recall and false positive rate, to see what the regularization trades off.

In [ ]:
plot_sweep(results, "LR T2", FIGS / "curve_lr_t2_pca.png")

> **[Your input: regularization]** Which `C` gives the best validation F1? Where do you see overfitting (big gap) or underfitting (both low)? How do precision, recall and FPR change? Send me your notes and I'll write them up.

**6. The best setting**

We pick the `C` with the highest validation F1. If it is at the edge of the grid, the best value might be outside it, and we should consider extending the grid.

In [ ]:
best = results.loc[results["val_f1"].idxmax()]
best_model = fitted[best["value"]]
print(f"best C = {best['value']:g}: val F1 = {best['val_f1']:.4f}, precision = {best['val_precision']:.4f}, "
      f"recall = {best['val_recall']:.4f}, FPR = {best['val_fpr']:.4f}")
if best["value"] in (GRID[0], GRID[-1]):
    print("the best value is at the edge of the grid ---> consider extending it")

The confusion matrix on the validation set: the top-right cell is false alarms (normal flagged as attack), the bottom-left cell is missed attacks.

In [ ]:
pred_val = best_model.predict(X_val)
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["Normal", "Attack"], cmap="Blues", values_format=",")
plt.title("LR T2, best C: validation")
plt.savefig(FIGS / "confusion_lr_t2_pca.png", dpi=150, bbox_inches="tight")
plt.show()

Which attack types does the best model miss? (On validation only, the test set stays untouched.)

In [ ]:
attack_recall(y_val, pred_val, cat_val)

> **[Your input: best model]** Which attack types are missed the most? Is that what we expected from the EDA (e.g. Fuzzers looking like normal traffic)? Send me your notes and I'll write them up.

**7. Save the results**

We save the 6 rows to `results/lr_t2_pca.csv`, so they can be merged with the other notebooks into the 72-row results table.

In [ ]:
results.to_csv(RESULTS / "lr_t2_pca.csv", index=False)
print("saved", len(results), "rows to", RESULTS / "lr_t2_pca.csv")